<a href="https://colab.research.google.com/github/Alline2312/Alline2312/blob/main/RESENHA2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# WHATSAPP RETROSPECTIVA — CÓDIGO COMPLETO (A + B + C)
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from openpyxl import Workbook

# -----------------------------
# PARTE 0 — CONFIGURAÇÕES
# -----------------------------
ARQUIVO_ORIGINAL = Path("_chat.txt")

ANO_MINIMO = 2026               # somente 2025+
MAX_RESP_MIN = 120              # resposta rápida: até 120 min
SILENCIO_MIN = 180              # revive grupo: silêncio >= 180 min

AUTORES_REMOVER = [
    "Você",
    "Meta AI",
    "Resenha 2.0 - Amor e Armênia",
    "Alline Vivo"
]

PALAVROES_LISTA = [
    "porra","caralho","krl","krlh","merda","puta","vtnc","fdp",
    "desgraça","desgraca","pqp","foda","fodase","foda-se","cu",
    "bosta","inferno","diabo","arrombado","arrombada", "buceta", "bct", "pau", "piroca", "canalha"
]

STOPWORDS = set("""
a o os as um uma uns umas de da do das dos em no na nos nas para por com sem
e ou que pq porque pra pro vc vcs ce cê eu tu ele ela eles elas meu minha
meus minhas seu sua seus suas nosso nossa nossos nossas vocês voces ja já
não nao sim ai aí la lá aqui ali umm hmm tipo mano mto mt muito muita tb tmb
então entao aff
""".split())

LIXO_TOKENS = {
    "figurinha","omitida","mensagem","apagada","fixou",
    "imagem","video","vídeo","áudio","audio","enquete",
    "opcao","opção","votos","voto"
}

SAIDA_EXCEL = Path("whatsapp_retrospectiva_2025_mais_COMPLETA.xlsx")
PASTA_GRAFICOS = Path("graficos")
PASTA_GRAFICOS.mkdir(exist_ok=True)

# -----------------------------
# PARTE 1 — PARSER ROBUSTO (ANTI COLAGEM)
# -----------------------------
INICIO_MSG = re.compile(r"[\u200e\u200f\u202a-\u202e]*\[\d{2}/\d{2}/\d{4}, \d{2}:\d{2}:\d{2}\]")
CABECALHO = re.compile(
    r'^[\u200e\u200f\u202a-\u202e]*\[(\d{2}/\d{2}/\d{4}), (\d{2}:\d{2}:\d{2})\] (.*?): (.*)$',
    flags=re.DOTALL
)

def split_mensagens(texto: str):
    texto = texto.replace("\r\n", "\n").replace("\r", "\n").strip()
    blocos = re.split(r"(?=" + INICIO_MSG.pattern + r")", texto)
    return [b.strip() for b in blocos if b.strip()]

def parse_whatsapp_robusto(arquivo: Path) -> pd.DataFrame:
    texto = arquivo.read_text(encoding="utf-8", errors="ignore")
    blocos = split_mensagens(texto)

    registros = []
    for b in blocos:
        m = CABECALHO.match(b)
        if not m:
            continue
        d, h, autor, msg = m.groups()
        dt = pd.to_datetime(f"{d} {h}", format="%d/%m/%Y %H:%M:%S", errors="coerce")
        registros.append([dt, autor.strip(), msg.strip()])

    df_ = pd.DataFrame(registros, columns=["datetime","autor","texto"]).dropna(subset=["datetime"])
    df_ = df_.sort_values("datetime").reset_index(drop=True)
    return df_

df = parse_whatsapp_robusto(ARQUIVO_ORIGINAL)
print("✅ Parser OK | Linhas:", len(df), "| Autores:", df["autor"].nunique())
# -----------------------------
# PARTE 2.0 — NORMALIZA AUTORES (remove emoji + unifica nomes)
# -----------------------------

EMOJI_NO_NOME_RE = re.compile(
    "[" +
    "\U0001F300-\U0001F5FF" +
    "\U0001F600-\U0001F64F" +
    "\U0001F680-\U0001F6FF" +
    "\U0001F700-\U0001F77F" +
    "\U0001F780-\U0001F7FF" +
    "\U0001F800-\U0001F8FF" +
    "\U0001F900-\U0001F9FF" +
    "\U0001FA00-\U0001FAFF" +
    "]+",
    flags=re.UNICODE
)

def limpa_nome_autor(nome: str) -> str:
    if not isinstance(nome, str):
        return nome
    # remove caracteres invisíveis do Whats
    nome = re.sub(r"[\u200e\u200f\u202a-\u202e]", "", nome)
    # remove emojis
    nome = EMOJI_NO_NOME_RE.sub("", nome)
    # remove símbolos soltos no início/fim
    nome = re.sub(r"^[\s\-\–\—\|\•\·\_]+|[\s\-\–\—\|\•\·\_]+$", "", nome)
    # normaliza espaços
    nome = re.sub(r"\s+", " ", nome).strip()
    return nome

df["autor"] = df["autor"].astype(str).apply(limpa_nome_autor)

# Mapeamento final (controle total)
MAP_AUTORES = {
    "Alline": "Alline Silveira",
    "andressa": "Andressa",
    "Amor": "Amor",
    "Yago": "Yago",
}

df["autor"] = df["autor"].replace(MAP_AUTORES)

print("✅ Autores normalizados:")
print(sorted(df["autor"].unique()))



# -----------------------------
# PARTE 2.0 — NORMALIZA AUTORES (mesma pessoa com nomes diferentes)
# -----------------------------
MAP_AUTORES = {
    "Alline": "Alline Silveira",
    # se existirem outras variações, adicione aqui:
    # "Alline S.": "Alline Silveira",
    # "Alline ✨": "Alline Silveira",
}

# Normalização leve: remove espaços duplicados e caracteres invisíveis comuns do Whats
df["autor"] = (
    df["autor"]
      .astype(str)
      .str.replace(r"[\u200e\u200f\u202a-\u202e]", "", regex=True)  # remove chars invisíveis
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# Aplica o mapeamento (se não estiver no dict, mantém o original)
df["autor"] = df["autor"].replace(MAP_AUTORES)

print("✅ Normalização autores aplicada.")
print("Autores após normalização:", df["autor"].nunique())

# -----------------------------
# PARTE 2 — FILTROS (AUTORES + 2025+)
# -----------------------------
df = df[~df["autor"].isin(AUTORES_REMOVER)].copy()
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df = df[df["datetime"].dt.year >= ANO_MINIMO].copy()
df = df.sort_values("datetime").reset_index(drop=True)

print("✅ Filtros OK")
print("Período:", df["datetime"].min(), "→", df["datetime"].max())
print("Anos presentes:", sorted(df["datetime"].dropna().dt.year.unique().tolist()))
print("Total mensagens:", len(df))
print("Autores restantes:", df["autor"].nunique())

# -----------------------------
# PARTE 3 — FUNÇÕES BASE
# -----------------------------
RISADA_RE = re.compile(r"(k{2,}|rs{2,}|ha(ha)+|he(he)+)", re.I)
K_RE = re.compile(r"(k{2,})", re.I)

def normaliza_tokens(texto: str):
    if not isinstance(texto, str):
        return []
    t = texto.lower()
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"@\S+", " ", t)
    t = re.sub(r"[^\w\sáàâãéèêíìîóòôõúùûç-]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    toks = [w for w in t.split() if w and (w not in STOPWORDS) and (len(w) >= 3) and (not w.isdigit())]
    return toks

def conta_risadas(texto: str) -> int:
    if not isinstance(texto, str):
        return 0
    return len(RISADA_RE.findall(texto))

def faixa_horaria(h: int) -> str:
    if 0 <= h <= 5: return "Madrugada (00-05)"
    if 6 <= h <= 11: return "Manhã (06-11)"
    if 12 <= h <= 17: return "Tarde (12-17)"
    return "Noite (18-23)"

pal_re = re.compile(r"\b(" + "|".join(map(re.escape, PALAVROES_LISTA)) + r")\b", flags=re.IGNORECASE)
def extrair_palavroes(texto: str):
    if not isinstance(texto, str):
        return []
    return pal_re.findall(texto.lower())

def tamanhos_k(texto: str):
    if not isinstance(texto, str):
        return []
    return [len(m) for m in K_RE.findall(texto.lower())]

EMOJI_RE = re.compile(
    "[" +
    "\U0001F300-\U0001F5FF" +
    "\U0001F600-\U0001F64F" +
    "\U0001F680-\U0001F6FF" +
    "\U0001F700-\U0001F77F" +
    "\U0001F780-\U0001F7FF" +
    "\U0001F800-\U0001F8FF" +
    "\U0001F900-\U0001F9FF" +
    "\U0001FA00-\U0001FAFF" +
    "]+",
    flags=re.UNICODE
)

# -----------------------------
# PARTE 4 — FEATURES (COLUNAS)
# -----------------------------
df["data"] = df["datetime"].dt.date
df["mes"] = df["datetime"].dt.to_period("M").astype(str)
df["hora"] = df["datetime"].dt.hour
df["dia_semana"] = df["datetime"].dt.day_name()
df["dia_mes"] = df["datetime"].dt.day
df["faixa"] = df["hora"].apply(faixa_horaria)

df["len_chars"] = df["texto"].astype(str).str.len()
df["tem_link"] = df["texto"].astype(str).str.contains(r"https?://", regex=True, na=False)

df["risadas_qtd"] = df["texto"].apply(conta_risadas)
df["tokens"] = df["texto"].apply(normaliza_tokens)

df["pal_encontrados"] = df["texto"].apply(extrair_palavroes)
df["palavroes_qtd"] = df["pal_encontrados"].apply(len)

df["k_sizes"] = df["texto"].apply(tamanhos_k)
df["emojis"] = df["texto"].astype(str).apply(lambda x: EMOJI_RE.findall(x))

tlow = df["texto"].astype(str).str.lower()
df["audio"] = tlow.str.contains("áudio omitido|audio omitido", regex=True, na=False)
df["video"] = tlow.str.contains("vídeo omitido|video omitido", regex=True, na=False)
df["figurinha"] = tlow.str.contains("figurinha omitida", regex=True, na=False)
df["foto"] = tlow.str.contains("imagem ocultada|imagem omitida|foto omitida", regex=True, na=False)

# -----------------------------
# PARTE 5 — RESPOSTAS RÁPIDAS + REVIVE GRUPO
# -----------------------------
df["dt_prev"] = df["datetime"].shift(1)
df["autor_prev"] = df["autor"].shift(1)
df["gap_min"] = (df["datetime"] - df["dt_prev"]).dt.total_seconds() / 60

resp = df[
    (df["autor"] != df["autor_prev"]) &
    (df["gap_min"].notna()) &
    (df["gap_min"] >= 0) &
    (df["gap_min"] <= MAX_RESP_MIN)
].copy()

resp_stats = (
    resp.groupby("autor")["gap_min"]
        .agg(respostas="count", media_min="mean", mediana_min="median", p90_min=lambda x: x.quantile(0.9))
        .reset_index()
)
resp_stats[["media_min","mediana_min","p90_min"]] = resp_stats[["media_min","mediana_min","p90_min"]].round(2)
resp_stats = resp_stats.sort_values(["mediana_min","respostas"], ascending=[True, False])

resp_ate5 = (
    resp.assign(ate5=resp["gap_min"] <= 5)
        .groupby("autor")["ate5"].sum()
        .reset_index(name="respostas_ate_5min")
        .sort_values("respostas_ate_5min", ascending=False)
)

revive = df[df["gap_min"] >= SILENCIO_MIN].copy()
revive_top = revive.groupby("autor").size().sort_values(ascending=False).reset_index(name="reviveu_qtd")

revive_recorde = (
    revive.groupby("autor")["gap_min"]
          .agg(revives="count", maior_silencio="max", media_silencio="mean")
          .reset_index()
)
revive_recorde[["maior_silencio","media_silencio"]] = revive_recorde[["maior_silencio","media_silencio"]].round(1)
revive_recorde = revive_recorde.sort_values(["revives","maior_silencio"], ascending=[False, False])

# -----------------------------
# PARTE 6 — RANKINGS BÁSICOS
# -----------------------------
ranking_mensagens = df.groupby("autor").size().sort_values(ascending=False).reset_index(name="mensagens")
ranking_palavroes = df.groupby("autor")["palavroes_qtd"].sum().sort_values(ascending=False).reset_index(name="palavroes_total")
ranking_risadas  = df.groupby("autor")["risadas_qtd"].sum().sort_values(ascending=False).reset_index(name="risadas_total")
links_rank = df.groupby("autor")["tem_link"].sum().sort_values(ascending=False).reset_index(name="links")

midia_rank = (
    df.groupby("autor")[["audio","video","figurinha","foto"]].sum()
      .sort_values(["figurinha","foto","video","audio"], ascending=False)
      .reset_index()
)

hora_mais_ativa = (
    df.groupby(["autor","hora"]).size()
      .reset_index(name="msgs")
      .sort_values(["autor","msgs"], ascending=[True, False])
      .drop_duplicates("autor")
      .sort_values("msgs", ascending=False)
)

distrib_faixa = (
    df.groupby(["autor","faixa"]).size()
      .reset_index(name="msgs")
      .sort_values(["autor","msgs"], ascending=[True, False])
)

corujas = (
    (df["faixa"].str.contains("Madrugada"))
    .groupby(df["autor"]).mean()
    .mul(100).round(1)
    .reset_index(name="%_madrugada")
    .sort_values("%_madrugada", ascending=False)
)

# -----------------------------
# PARTE 7 — EMOJIS
# -----------------------------
emoji_df = (
    df[["autor","emojis"]]
      .explode("emojis")
      .dropna(subset=["emojis"])
      .rename(columns={"emojis":"emoji"})
)
# -----------------------------
# PARTE 7.1 — REMOVER EMOJIS DE NOME / DECORAÇÃO
# -----------------------------
EMOJIS_EXCLUIR = {"💚", "🧡", "📓"}

emoji_df = emoji_df[~emoji_df["emoji"].isin(EMOJIS_EXCLUIR)].copy()

print("✅ Emojis removidos da contagem:", EMOJIS_EXCLUIR)

top_emojis_grupo = (
    emoji_df.groupby("emoji").size()
    .sort_values(ascending=False)
    .reset_index(name="qtd")
)

top_emojis_pessoa = (
    emoji_df.groupby(["autor","emoji"]).size()
    .reset_index(name="qtd")
    .sort_values(["autor","qtd"], ascending=[True, False])
    .groupby("autor").head(5)
)

# -----------------------------
# PARTE 8 — MÉDIA DE K NO KKKKK
# -----------------------------
k_df = (
    df[["autor","k_sizes"]]
    .explode("k_sizes")
    .dropna(subset=["k_sizes"])
    .rename(columns={"k_sizes":"num_k"})
)

k_stats_pessoa = (
    k_df.groupby("autor")["num_k"]
    .agg(risadas_com_k="count", media_k="mean", mediana_k="median", max_k="max")
    .reset_index()
)
k_stats_pessoa[["media_k","mediana_k","max_k"]] = k_stats_pessoa[["media_k","mediana_k","max_k"]].round(1)
ranking_media_k = k_stats_pessoa.sort_values("media_k", ascending=False)

recorde_k = (
    df.explode("k_sizes").dropna(subset=["k_sizes"])
      .sort_values("k_sizes", ascending=False)
      .head(1)[["datetime","autor","texto","k_sizes"]]
      .rename(columns={"k_sizes":"recorde_num_k"})
)

# -----------------------------
# PARTE 9 — MENSAGENS POR MÊS + DIAS MAIS FALADOS
# -----------------------------
msgs_por_mes = (
    df.groupby("mes").size()
      .reset_index(name="mensagens")
      .sort_values("mes")
)

msgs_por_dia = (
    df.groupby("data").size()
      .reset_index(name="mensagens")
      .sort_values("mensagens", ascending=False)
)

top_dias = msgs_por_dia.head(15).copy()

msgs_por_dia_semana = (
    df.groupby("dia_semana").size()
      .reset_index(name="mensagens")
      .sort_values("mensagens", ascending=False)
)

msgs_por_dia_mes = (
    df.groupby("dia_mes").size()
      .reset_index(name="mensagens")
      .sort_values("mensagens", ascending=False)
)

# -----------------------------
# PARTE 10 — TOP PALAVRAS
# -----------------------------
words = df.explode("tokens").dropna(subset=["tokens"]).copy()
words = words[~words["tokens"].isin(LIXO_TOKENS)]

top20_grupo = words["tokens"].value_counts().head(20).reset_index()
top20_grupo.columns = ["token","qtd"]

top10_palavras_por_pessoa = (
    words.groupby(["autor","tokens"]).size()
         .reset_index(name="qtd")
         .sort_values(["autor","qtd"], ascending=[True, False])
         .groupby("autor").head(10)
)

top_palavras_mes = (
    words.groupby(["mes","tokens"]).size()
         .reset_index(name="qtd")
         .sort_values(["mes","qtd"], ascending=[True, False])
         .groupby("mes").head(10)
)

# -----------------------------
# PARTE 11 — CAMPEÃO DO MÊS
# -----------------------------
msgs_mes_pessoa = (
    df.groupby(["mes","autor"]).size()
      .reset_index(name="mensagens")
      .sort_values(["mes","mensagens"], ascending=[True, False])
)
campeao_do_mes = msgs_mes_pessoa.groupby("mes").head(1).reset_index(drop=True)

# -----------------------------
# PARTE 12 — TEXTÃO / MSG MAIS LONGA
# -----------------------------
msg_mais_longa_grupo = (
    df.sort_values("len_chars", ascending=False)
      .head(1)[["datetime","autor","len_chars","texto"]]
)
msg_mais_longa_por_pessoa = (
    df.sort_values("len_chars", ascending=False)
      .groupby("autor", as_index=False)
      .head(1)[["autor","datetime","len_chars","texto"]]
      .sort_values("len_chars", ascending=False)
)
estilo_msg = (
    df.groupby("autor")["len_chars"]
      .agg(msgs="count", media_chars="mean", mediana_chars="median", p90_chars=lambda x: x.quantile(0.9), max_chars="max")
      .reset_index()
)
estilo_msg[["media_chars","mediana_chars","p90_chars","max_chars"]] = estilo_msg[["media_chars","mediana_chars","p90_chars","max_chars"]].round(1)

# -----------------------------
# PARTE 13 — PALAVRÃO FAVORITO POR PESSOA
# -----------------------------
pal_df = (
    df[["autor","pal_encontrados"]]
      .explode("pal_encontrados")
      .dropna(subset=["pal_encontrados"])
      .rename(columns={"pal_encontrados":"palavrao"})
)

ranking_total_pal = (
    pal_df.groupby("autor").size()
          .sort_values(ascending=False)
          .reset_index(name="palavroes_total")
)

contagem_por_tipo = (
    pal_df.groupby(["autor","palavrao"]).size()
          .reset_index(name="qtd")
          .sort_values(["autor","qtd"], ascending=[True, False])
)

favorito_por_pessoa = (
    contagem_por_tipo
      .sort_values(["autor","qtd"], ascending=[True, False])
      .drop_duplicates("autor")
      .rename(columns={"palavrao":"palavrao_favorito","qtd":"qtd_favorito"})
)

ranking_palavroes_com_favorito = ranking_total_pal.merge(favorito_por_pessoa, on="autor", how="left")
ranking_palavroes_com_favorito["%_favorito"] = (
    ranking_palavroes_com_favorito["qtd_favorito"] / ranking_palavroes_com_favorito["palavroes_total"] * 100
).round(1)

top10_palavroes_favorito = ranking_palavroes_com_favorito.head(10).copy()

# -----------------------------
# PARTE 14 — PARTICIPAÇÃO (TAXA DE RESPOSTA PROXY)
# -----------------------------
total_msgs = df.groupby("autor").size().rename("msgs")
respostas = resp.groupby("autor").size().rename("respostas")
participacao = pd.concat([total_msgs, respostas], axis=1).fillna(0).reset_index()
participacao["taxa_resposta_%"] = (participacao["respostas"] / participacao["msgs"] * 100).round(1)
participacao = participacao.sort_values("taxa_resposta_%", ascending=True)

# ============================================================
# ========================== PARTE A ==========================
# ============================================================
def pct_upper(s: str) -> float:
    if not isinstance(s, str) or not s:
        return 0.0
    letters = [ch for ch in s if ch.isalpha()]
    if not letters:
        return 0.0
    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters)

def count_exclaim(s: str) -> int:
    return len(re.findall(r"!", s)) if isinstance(s, str) else 0

def count_question(s: str) -> int:
    return len(re.findall(r"\?", s)) if isinstance(s, str) else 0

def count_multi(s: str, pattern: str) -> int:
    return len(re.findall(pattern, s)) if isinstance(s, str) else 0

def contains_words(s: str, words):
    if not isinstance(s, str):
        return False
    t = s.lower()
    return any(w in t for w in words)

# A1) Streak do mesmo autor
df["mesmo_autor_prev"] = df["autor"] == df["autor_prev"]
streak = []
cur = 0
for same in df["mesmo_autor_prev"].fillna(False).to_list():
    cur = cur + 1 if same else 1
    streak.append(cur)
df["streak_autor"] = streak

maior_streak_por_pessoa = (
    df.groupby("autor")["streak_autor"].max()
      .reset_index(name="maior_sequencia_msgs")
      .sort_values("maior_sequencia_msgs", ascending=False)
)

monologo_linhas = df[df["streak_autor"] >= 3].copy()
monologo_stats = (
    monologo_linhas.groupby("autor").size()
    .reset_index(name="msgs_em_monologo_(streak>=3)")
    .sort_values("msgs_em_monologo_(streak>=3)", ascending=False)
)

# A2) Caps lock
df["pct_maiusculas"] = df["texto"].astype(str).apply(pct_upper)
df["gritou"] = (df["pct_maiusculas"] >= 0.6) & (df["len_chars"] >= 10)
ranking_gritaria = (
    df.groupby("autor")["gritou"].sum()
      .reset_index(name="qtd_gritos_(>=60%caps)")
      .sort_values("qtd_gritos_(>=60%caps)", ascending=False)
)

# A3) Drama
df["excl_qtd"] = df["texto"].astype(str).apply(count_exclaim)
df["inter_qtd"] = df["texto"].astype(str).apply(count_question)
df["triple_excl"] = df["texto"].astype(str).apply(lambda s: count_multi(s, r"!{3,}"))
df["triple_inter"] = df["texto"].astype(str).apply(lambda s: count_multi(s, r"\?{3,}"))

ranking_drama = (
    df.groupby("autor")[["excl_qtd","inter_qtd","triple_excl","triple_inter"]].sum()
      .reset_index()
)
ranking_drama["drama_score"] = ranking_drama["triple_excl"] + ranking_drama["triple_inter"] + (ranking_drama["excl_qtd"] + ranking_drama["inter_qtd"]) / 10
ranking_drama = ranking_drama.sort_values("drama_score", ascending=False)

# A4) RH do grupo
bom_dia_words = ["bom dia", "bomdia"]
boa_noite_words = ["boa noite", "boanoite"]
boa_tarde_words = ["boa tarde", "boatarde"]

df["bom_dia"] = df["texto"].astype(str).str.lower().apply(lambda s: contains_words(s, bom_dia_words))
df["boa_tarde"] = df["texto"].astype(str).str.lower().apply(lambda s: contains_words(s, boa_tarde_words))
df["boa_noite"] = df["texto"].astype(str).str.lower().apply(lambda s: contains_words(s, boa_noite_words))

ranking_rh = (
    df.groupby("autor")[["bom_dia","boa_tarde","boa_noite"]].sum()
      .reset_index()
)
ranking_rh["total_cumprimentos"] = ranking_rh["bom_dia"] + ranking_rh["boa_tarde"] + ranking_rh["boa_noite"]
ranking_rh = ranking_rh.sort_values("total_cumprimentos", ascending=False)

# A5) Sumido
df["dt_prev_mesmo_autor"] = df.groupby("autor")["datetime"].shift(1)
df["gap_autor_min"] = (df["datetime"] - df["dt_prev_mesmo_autor"]).dt.total_seconds() / 60

sumico_stats = (
    df.groupby("autor")["gap_autor_min"]
      .agg(maior_sumico_min="max", media_sumico_min="mean")
      .reset_index()
)
sumico_stats[["maior_sumico_min","media_sumico_min"]] = sumico_stats[["maior_sumico_min","media_sumico_min"]].round(1)
sumico_stats = sumico_stats.sort_values("maior_sumico_min", ascending=False)

# A6) Linkzeiro
ranking_links = (
    df.groupby("autor")["tem_link"].sum()
      .reset_index(name="links_total")
      .sort_values("links_total", ascending=False)
)

print("✅ PARTE A ok")

# ============================================================
# ========================== PARTE B ==========================
# ============================================================
resp_pairs = resp[["autor_prev","autor","gap_min"]].copy()
resp_pairs.columns = ["quem_falou_antes","quem_respondeu","tempo_resp_min"]

matriz_respostas = (
    resp_pairs.groupby(["quem_falou_antes","quem_respondeu"])
    .agg(qtd_respostas=("tempo_resp_min","count"),
         tempo_medio=("tempo_resp_min","mean"),
         tempo_mediano=("tempo_resp_min","median"))
    .reset_index()
)
matriz_respostas[["tempo_medio","tempo_mediano"]] = matriz_respostas[["tempo_medio","tempo_mediano"]].round(2)
matriz_respostas = matriz_respostas.sort_values("qtd_respostas", ascending=False)
top_duplas = matriz_respostas.head(20).copy()

eco_de = (
    matriz_respostas.sort_values(["quem_falou_antes","qtd_respostas"], ascending=[True, False])
    .groupby("quem_falou_antes").head(1)
    .reset_index(drop=True)
    .rename(columns={"quem_falou_antes":"pessoa","quem_respondeu":"eco_oficial","qtd_respostas":"qtd","tempo_mediano":"mediana_min"})
)

alvo_de = (
    matriz_respostas.sort_values(["quem_respondeu","qtd_respostas"], ascending=[True, False])
    .groupby("quem_respondeu").head(1)
    .reset_index(drop=True)
    .rename(columns={"quem_respondeu":"pessoa","quem_falou_antes":"alvo_favorito","qtd_respostas":"qtd","tempo_mediano":"mediana_min"})
)

# Riso contagioso (janela 2 min)
JANELA_RISO_MIN = 2
dt_arr = df["datetime"].to_numpy()
tem_riso = (df["risadas_qtd"] > 0).to_numpy()

contagios = []
for i in range(len(df)):
    if not tem_riso[i]:
        continue
    t0 = dt_arr[i]
    j = i + 1
    count = 0
    while j < len(df):
        delta_min = (dt_arr[j] - t0) / np.timedelta64(1, 'm')
        if delta_min > JANELA_RISO_MIN:
            break
        if tem_riso[j]:
            count += 1
        j += 1
    contagios.append((df.at[i,"autor"], count))

riso_contagioso = (
    pd.DataFrame(contagios, columns=["autor","risos_gerados_em_2min"])
    .groupby("autor")["risos_gerados_em_2min"]
    .agg(eventos_riso="count", media_risos_gerados="mean", total_risos_gerados="sum")
    .reset_index()
)
riso_contagioso[["media_risos_gerados"]] = riso_contagioso[["media_risos_gerados"]].round(2)
riso_contagioso = riso_contagioso.sort_values("total_risos_gerados", ascending=False)

# “Chega logo depois” (até 5 min)
JANELA_APARECE_MIN = 5
aparece_pairs = df[
    (df["autor"] != df["autor_prev"]) &
    (df["gap_min"].notna()) &
    (df["gap_min"] <= JANELA_APARECE_MIN)
][["autor_prev","autor","gap_min"]].copy()
aparece_pairs.columns = ["A","B","gap_min"]

chega_depois = (
    aparece_pairs.groupby(["A","B"]).size()
    .reset_index(name=f"qtd_em_ate_{JANELA_APARECE_MIN}min")
    .sort_values(f"qtd_em_ate_{JANELA_APARECE_MIN}min", ascending=False)
)

print("✅ PARTE B ok")

# ============================================================
# ========================== PARTE C ==========================
# ============================================================
def top1(df_, col_val, ascending=False):
    if df_ is None or len(df_) == 0:
        return None
    return df_.sort_values(col_val, ascending=ascending).head(1)

def fmt_min_to_h(mins):
    if pd.isna(mins):
        return ""
    mins = float(mins)
    h = int(mins // 60)
    m = int(round(mins % 60))
    if h <= 0:
        return f"{m} min"
    return f"{h}h {m:02d}min"

# Perfil do personagem
emoji_fav = (
    top_emojis_pessoa.sort_values(["autor","qtd"], ascending=[True, False])
    .drop_duplicates("autor")[["autor","emoji","qtd"]]
    .rename(columns={"emoji":"emoji_favorito","qtd":"emoji_fav_qtd"})
)

pal_fav = ranking_palavroes_com_favorito[["autor","palavrao_favorito","qtd_favorito","palavroes_total","%_favorito"]].copy()
k_med = ranking_media_k[["autor","media_k","max_k","risadas_com_k"]].copy()
hora_fav = hora_mais_ativa[["autor","hora","msgs"]].rename(columns={"hora":"hora_mais_ativa","msgs":"msgs_na_hora_fav"})
rev_cnt = revive_top.rename(columns={"reviveu_qtd":"revives"}).copy()
est = estilo_msg[["autor","media_chars","p90_chars","max_chars"]].copy()

perfil = (
    ranking_mensagens.merge(emoji_fav, on="autor", how="left")
    .merge(pal_fav, on="autor", how="left")
    .merge(k_med, on="autor", how="left")
    .merge(hora_fav, on="autor", how="left")
    .merge(rev_cnt, on="autor", how="left")
    .merge(est, on="autor", how="left")
)
perfil["revives"] = perfil["revives"].fillna(0).astype(int)
perfil = perfil.sort_values("mensagens", ascending=False)

# Prêmios zoeira
premios = []

def add_premio(nome, vencedor, detalhe, texto_slide):
    premios.append({
        "premio": nome,
        "vencedor": vencedor,
        "detalhe": detalhe,
        "texto_slide_pronto": texto_slide
    })

premio_mais_fala = top1(ranking_mensagens, "mensagens", ascending=False)
premio_mais_palavrao = top1(ranking_palavroes_com_favorito, "palavroes_total", ascending=False)
premio_mais_riso = top1(ranking_risadas, "risadas_total", ascending=False)
premio_linkzeiro = top1(ranking_links, "links_total", ascending=False)
premio_gritaria = top1(ranking_gritaria, "qtd_gritos_(>=60%caps)", ascending=False)
premio_drama = top1(ranking_drama, "drama_score", ascending=False)
premio_rh = top1(ranking_rh, "total_cumprimentos", ascending=False)
premio_monologo = top1(maior_streak_por_pessoa, "maior_sequencia_msgs", ascending=False)
premio_sumido = top1(sumico_stats, "maior_sumico_min", ascending=False)
premio_revive = top1(revive_top, "reviveu_qtd", ascending=False)
premio_riso_infinito = top1(ranking_media_k, "media_k", ascending=False)

emoji_oficial = top_emojis_grupo.head(1).copy()
dia_do_caos = top_dias.head(1).copy()
msgs_por_hora = df.groupby("hora").size().reset_index(name="mensagens").sort_values("mensagens", ascending=False)
hora_proibida = msgs_por_hora.head(1).copy()
dupla_1 = top_duplas.head(1).copy()
riso_cont_top = riso_contagioso.head(1).copy()

if premio_mais_fala is not None:
    a = premio_mais_fala.iloc[0]
    add_premio("👑 Quem mais fala", a["autor"], f'{int(a["mensagens"])} mensagens',
               f'👑 **{a["autor"]}** dominou o grupo com **{int(a["mensagens"])} mensagens**. Fala até dormindo.')

if premio_mais_palavrao is not None:
    a = premio_mais_palavrao.iloc[0]
    fav = a.get("palavrao_favorito","")
    add_premio("🤬 Boca suja oficial", a["autor"],
               f'{int(a["palavroes_total"])} palavrões | favorito: {fav}',
               f'🤬 **{a["autor"]}** foi o terror do dicionário: **{int(a["palavroes_total"])} palavrões**. Bordão: **{fav}**.')

if premio_mais_riso is not None:
    a = premio_mais_riso.iloc[0]
    add_premio("🤣 Palhaço(a) do grupo", a["autor"], f'{int(a["risadas_total"])} risadas',
               f'🤣 **{a["autor"]}** manteve a sanidade do grupo com **{int(a["risadas_total"])} risadas**.')

if len(emoji_oficial):
    e = emoji_oficial.iloc[0]
    add_premio("😂 Emoji oficial do grupo", e["emoji"], f'{int(e["qtd"])} usos',
               f'😂 O emoji oficial do grupo foi **{e["emoji"]}** com **{int(e["qtd"])}** aparições. Isso diz muito…')

if premio_riso_infinito is not None:
    a = premio_riso_infinito.iloc[0]
    add_premio("📏 Riso infinito (média de K)", a["autor"], f'média: {a["media_k"]} | máx: {a["max_k"]}',
               f'📏 **{a["autor"]}** não ri… digita **parágrafo**. Média: **{a["media_k"]} k** | recorde pessoal: **{a["max_k"]} k**.')

if len(recorde_k):
    r = recorde_k.iloc[0]
    add_premio("🏆 Recorde absoluto de K", r["autor"], f'{int(r["recorde_num_k"])} k',
               f'🏆 **{r["autor"]}** atingiu o recorde mundial do grupo: **{int(r["recorde_num_k"])} k** numa risada só. Cientistas em choque.')

if len(msg_mais_longa_grupo):
    t = msg_mais_longa_grupo.iloc[0]
    add_premio("📜 Textão do ano", t["autor"], f'{int(t["len_chars"])} caracteres',
               f'📜 **{t["autor"]}** escreveu um TCC no grupo: **{int(t["len_chars"])} caracteres**. ABNT aprova.')

if premio_monologo is not None:
    a = premio_monologo.iloc[0]
    add_premio("🎤 Monólogo", a["autor"], f'sequência máx: {int(a["maior_sequencia_msgs"])} msgs seguidas',
               f'🎤 **{a["autor"]}** venceu o prêmio **Monólogo**: mandou **{int(a["maior_sequencia_msgs"])} mensagens seguidas** sem respirar.')

if premio_revive is not None and len(premio_revive):
    a = premio_revive.iloc[0]
    add_premio("🧟 Revive o grupo", a["autor"], f'{int(a["reviveu_qtd"])} revives (silêncio ≥ {SILENCIO_MIN} min)',
               f'🧟 **{a["autor"]}** é o ressuscitador oficial: reviveu o grupo **{int(a["reviveu_qtd"])} vezes** depois de silêncio mortal.')

if premio_sumido is not None:
    a = premio_sumido.iloc[0]
    add_premio("👻 O sumido", a["autor"], f'maior sumiço: {fmt_min_to_h(a["maior_sumico_min"])}',
               f'👻 **{a["autor"]}** sumiu por **{fmt_min_to_h(a["maior_sumico_min"])}** e voltou como se nada tivesse acontecido.')

if premio_linkzeiro is not None:
    a = premio_linkzeiro.iloc[0]
    add_premio("🔗 Linkzeiro(a)", a["autor"], f'{int(a["links_total"])} links',
               f'🔗 **{a["autor"]}** foi o curador do caos: **{int(a["links_total"])} links** (metade suspeito).')

if premio_gritaria is not None:
    a = premio_gritaria.iloc[0]
    add_premio("🔊 Caps Lock", a["autor"], f'{int(a["qtd_gritos_(>=60%caps)"])} gritos',
               f'🔊 **{a["autor"]}** gritou no teclado **{int(a["qtd_gritos_(>=60%caps)"])} vezes**. Acalma, diva.')

if premio_drama is not None:
    a = premio_drama.iloc[0]
    add_premio("‼️ Drama Queen/King", a["autor"],
               f'!!!: {int(a["triple_excl"])} | ???: {int(a["triple_inter"])}',
               f'‼️ **{a["autor"]}** ganhou no drama: **{int(a["triple_excl"])}** “!!!” e **{int(a["triple_inter"])}** “???” — terapia, talvez.')

if premio_rh is not None:
    a = premio_rh.iloc[0]
    add_premio("☀️ RH do grupo", a["autor"], f'{int(a["total_cumprimentos"])} cumprimentos',
               f'☀️ **{a["autor"]}** é o RH: mandou **{int(a["total_cumprimentos"])}** “bom dia/boa tarde/boa noite”.')

if len(dia_do_caos):
    d = dia_do_caos.iloc[0]
    add_premio("🔥 Dia do caos", str(d["data"]), f'{int(d["mensagens"])} mensagens',
               f'🔥 O **dia do caos** foi **{d["data"]}** com **{int(d["mensagens"])} mensagens**. A moderação chorou.')

if len(hora_proibida):
    h = hora_proibida.iloc[0]
    add_premio("⛔ Horário proibido", f'{int(h["hora"])}h', f'{int(h["mensagens"])} mensagens',
               f'⛔ O horário proibido é **{int(h["hora"])}h**: **{int(h["mensagens"])} mensagens**. Nesse horário, o bom senso dorme.')

if len(dupla_1):
    x = dupla_1.iloc[0]
    add_premio("🤝 Dupla dinâmica", f'{x["quem_falou_antes"]} → {x["quem_respondeu"]}', f'{int(x["qtd_respostas"])} interações',
               f'🤝 A dupla dinâmica foi **{x["quem_falou_antes"]}** e **{x["quem_respondeu"]}**: **{int(x["qtd_respostas"])}** respostas rápidas.')

if len(riso_cont_top):
    x = riso_cont_top.iloc[0]
    add_premio("🦠 Riso contagioso", x["autor"], f'total gerado: {int(x["total_risos_gerados"])}',
               f'🦠 **{x["autor"]}** espalhou risada: gerou **{int(x["total_risos_gerados"])}** risadas em cadeia.')

premios_df = pd.DataFrame(premios)
print("✅ PARTE C ok — Prêmios gerados:", len(premios_df))

# -----------------------------
# PARTE 15 — RESUMO
# -----------------------------
resumo = pd.DataFrame({
    "total_msgs":[len(df)],
    "participantes":[df["autor"].nunique()],
    "inicio":[df["datetime"].min()],
    "fim":[df["datetime"].max()],
    "ano_minimo":[ANO_MINIMO],
    "silencio_revive_min":[SILENCIO_MIN],
    "limite_resposta_min":[MAX_RESP_MIN],
})

# -----------------------------
# PARTE 16 — EXPORT EXCEL (ULTRA COMPATÍVEL)
# -----------------------------
def limpar_excel_extremo(x):
    if not isinstance(x, str):
        return x
    x = re.sub(r"[\x00-\x08\x0B\x0C\x0E-\x1F]", "", x)
    x = re.sub(r"[\u200e\u200f\u202a-\u202e]", "", x)
    return x

def limpar_df_extremo(d: pd.DataFrame) -> pd.DataFrame:
    out = d.copy()
    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].apply(limpar_excel_extremo)
    return out

def to_excel_safe_value(x):
    if isinstance(x, list):
        return ", ".join(map(str, x))
    if isinstance(x, dict):
        return str(x)
    return x

def add_sheet(wb: Workbook, nome: str, d: pd.DataFrame):
    ws = wb.create_sheet(title=nome[:31])
    d2 = d.copy()

    # datetime -> string
    for c in d2.columns:
        if np.issubdtype(d2[c].dtype, np.datetime64):
            d2[c] = d2[c].dt.strftime("%Y-%m-%d %H:%M:%S")

    # listas -> string
    for c in d2.columns:
        d2[c] = d2[c].apply(to_excel_safe_value)

    # limpeza extrema
    d2 = limpar_df_extremo(d2)

    ws.append(list(d2.columns))
    for row in d2.itertuples(index=False):
        ws.append(list(row))

wb = Workbook()
wb.remove(wb.active)

# Base completa pode ficar pesada, mas você pediu completo :)
add_sheet(wb, "Resumo", resumo)
add_sheet(wb, "Base_2025+", df)

add_sheet(wb, "Ranking_Mensagens", ranking_mensagens)
add_sheet(wb, "Ranking_Palavroes", ranking_palavroes)
add_sheet(wb, "Ranking_Risadas", ranking_risadas)
add_sheet(wb, "Links_Ranking", links_rank)
add_sheet(wb, "Midias_Ranking", midia_rank)
add_sheet(wb, "Hora_Mais_Ativa", hora_mais_ativa)
add_sheet(wb, "Distrib_Faixa", distrib_faixa)
add_sheet(wb, "Corujas_%Madrugada", corujas)

add_sheet(wb, "Msgs_Por_Mes", msgs_por_mes)
add_sheet(wb, "Msgs_Por_Dia", msgs_por_dia)
add_sheet(wb, "Top_Dias", top_dias)
add_sheet(wb, "Msgs_Dia_Semana", msgs_por_dia_semana)
add_sheet(wb, "Msgs_Dia_Mes", msgs_por_dia_mes)
add_sheet(wb, "Msgs_Mes_Pessoa", msgs_mes_pessoa)
add_sheet(wb, "Campeao_Do_Mes", campeao_do_mes)

add_sheet(wb, "Respostas_Rapidas", resp_stats)
add_sheet(wb, "Respostas_ate5min", resp_ate5)
add_sheet(wb, "Revive_Top", revive_top)
add_sheet(wb, "Revive_Recordes", revive_recorde)

add_sheet(wb, "Msg_Mais_Longa_Grupo", msg_mais_longa_grupo)
add_sheet(wb, "Msg_Mais_Longa_Pessoa", msg_mais_longa_por_pessoa)
add_sheet(wb, "Estilo_Msg", estilo_msg)

add_sheet(wb, "Top20_Palavras_Grupo", top20_grupo)
add_sheet(wb, "Top10_Palavras_Pessoa", top10_palavras_por_pessoa)
add_sheet(wb, "Top10_Palavras_Mes", top_palavras_mes)

add_sheet(wb, "Top_Emojis_Grupo", top_emojis_grupo.head(100))
add_sheet(wb, "Top_Emojis_Pessoa", top_emojis_pessoa)

add_sheet(wb, "K_Stats_Pessoa", k_stats_pessoa)
add_sheet(wb, "Ranking_Media_K", ranking_media_k)
add_sheet(wb, "Recorde_K", recorde_k)

add_sheet(wb, "Palavroes_Favorito", ranking_palavroes_com_favorito)
add_sheet(wb, "Palavroes_Por_Tipo", contagem_por_tipo)

add_sheet(wb, "Participacao_TaxaResposta", participacao)

# Parte A
add_sheet(wb, "A_Maior_Streak_Autor", maior_streak_por_pessoa)
add_sheet(wb, "A_Monologo_Stats", monologo_stats)
add_sheet(wb, "A_Ranking_Gritaria", ranking_gritaria)
add_sheet(wb, "A_Ranking_Drama", ranking_drama)
add_sheet(wb, "A_Ranking_RH", ranking_rh)
add_sheet(wb, "A_Sumico_Stats", sumico_stats)
add_sheet(wb, "A_Ranking_Links", ranking_links)

# Parte B
add_sheet(wb, "B_Matriz_Respostas", matriz_respostas.head(500))
add_sheet(wb, "B_Top_Duplas", top_duplas)
add_sheet(wb, "B_Eco_Oficial", eco_de)
add_sheet(wb, "B_Alvo_Favorito", alvo_de)
add_sheet(wb, "B_Riso_Contagioso", riso_contagioso)
add_sheet(wb, "B_Chega_Depois_5min", chega_depois.head(200))

# Parte C
add_sheet(wb, "C_Perfil_Pessoa", perfil)
add_sheet(wb, "C_Premios_Zoeira", premios_df)

wb.save(SAIDA_EXCEL)
print("✅ Excel COMPLETO gerado:", SAIDA_EXCEL.resolve())

# -----------------------------
# PARTE 17 — GRÁFICOS PNG PRO CANVA
# -----------------------------
# 17.1 Mensagens por mês
plt.figure()
plt.plot(msgs_por_mes["mes"], msgs_por_mes["mensagens"])
plt.xticks(rotation=45)
plt.title("Mensagens por mês (2025+)")
plt.xlabel("Mês")
plt.ylabel("Mensagens")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "mensagens_por_mes.png")
plt.close()

# 17.2 Top 10 dias mais falados
top10_dias_plot = msgs_por_dia.head(10)
plt.figure()
plt.bar(top10_dias_plot["data"].astype(str), top10_dias_plot["mensagens"])
plt.xticks(rotation=45)
plt.title("Top 10 dias mais falados (2025+)")
plt.xlabel("Dia")
plt.ylabel("Mensagens")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "top10_dias_mais_falados.png")
plt.close()

# 17.3 Dia da semana
ordem_semana = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dados_semana = msgs_por_dia_semana.set_index("dia_semana").reindex(ordem_semana).reset_index()

plt.figure()
plt.bar(dados_semana["dia_semana"], dados_semana["mensagens"])
plt.xticks(rotation=30)
plt.title("Mensagens por dia da semana (2025+)")
plt.xlabel("Dia da semana")
plt.ylabel("Mensagens")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "mensagens_por_dia_semana.png")
plt.close()

# 17.4 Hora do dia
msgs_por_hora_plot = df.groupby("hora").size().reset_index(name="mensagens")
plt.figure()
plt.bar(msgs_por_hora_plot["hora"], msgs_por_hora_plot["mensagens"])
plt.title("Mensagens por hora do dia (2025+)")
plt.xlabel("Hora")
plt.ylabel("Mensagens")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "mensagens_por_hora.png")
plt.close()

# 17.5 Top emojis
top10_emojis = top_emojis_grupo.head(10)
plt.figure()
plt.bar(top10_emojis["emoji"], top10_emojis["qtd"])
plt.title("Top 10 emojis (2025+)")
plt.xlabel("Emoji")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "top10_emojis.png")
plt.close()

# 17.6 Média de K
top_k = ranking_media_k.head(10)
plt.figure()
plt.bar(top_k["autor"], top_k["media_k"])
plt.xticks(rotation=45)
plt.title("Top 10 — Média de 'k' no kkkkk (2025+)")
plt.xlabel("Pessoa")
plt.ylabel("Média de k")
plt.tight_layout()
plt.savefig(PASTA_GRAFICOS / "media_k_por_pessoa.png")
plt.close()

ryan_msgs = df[df["autor"] == "Pedro Gomes - UFABC"].copy()
max_streak = ryan_msgs["streak_autor"].max()
print("Streak do Ryan:", max_streak)
monologo_ryan = ryan_msgs[ryan_msgs["streak_autor"] == max_streak]
dias_monologo = monologo_ryan["datetime"].dt.date.unique()
print("Dias em monólogo do Ryan:", dias_monologo)

print("✅ Gráficos PNG gerados em:", PASTA_GRAFICOS.resolve())
print("🎬 Aba pronta pro Canva: C_Premios_Zoeira (texto_slide_pronto)")


✅ Parser OK | Linhas: 85437 | Autores: 21
✅ Autores normalizados:
['Alline Silveira', 'Alline Vivo', 'Amor', 'Andressa', 'Bia', 'Bubu', 'Cecelia', 'Gabi Gomes', 'Gabi Silveira (Clone)', 'Marcos Marsola - UFABC', 'Meta AI', 'Pedro Cabeludo', 'Pedro Gomes - UFABC', 'Pedro Guaicurú', 'Pedro Irmao', 'Poliana Gomes', 'Resenha 2.0 - Amor e Armênia', 'Ryan Boy', 'Skyra (duda)', 'Você', 'Yago']
✅ Normalização autores aplicada.
Autores após normalização: 21
✅ Filtros OK
Período: 2026-01-01 00:03:13 → 2026-01-24 23:20:48
Anos presentes: [2026]
Total mensagens: 3349
Autores restantes: 16
✅ Emojis removidos da contagem: {'📓', '💚', '🧡'}
✅ PARTE A ok
✅ PARTE B ok
✅ PARTE C ok — Prêmios gerados: 18
✅ Excel COMPLETO gerado: /content/whatsapp_retrospectiva_2025_mais_COMPLETA.xlsx


/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 129001 (\N{LARGE GREEN SQUARE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 128646 (\N{TRAIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 129392 (\N{SMILING FACE WITH SMILING EYES AND THREE HEARTS}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 128148 (\N{BROKEN HEART}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 129000 (\N{LARGE YELLOW SQUARE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 128204 (\N{PUSHPIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-3533126980.py:1054: UserWarning: Glyph 129394 (\N{SMILING FACE WITH TEAR}) missing from font(s) DejaVu Sans.
  

Streak do Ryan: 5
Dias em monólogo do Ryan: [datetime.date(2026, 1, 22)]
✅ Gráficos PNG gerados em: /content/graficos
🎬 Aba pronta pro Canva: C_Premios_Zoeira (texto_slide_pronto)
